# Structured output — the contract (live demo)

Companion to the lecture **Structured output — the contract**. We treat
structured output as the typed object that sits between the LLM and the agent
runtime, and we exercise each clause of that contract:

| Deck part | Notebook section |
|---|---|
| 1 — The contract | §1 `chat.completions.parse` → a validated object |
| 2 — Routing | §2 `RouteDecision` |
| 2 — Planning | §3 `Plan` / `PlanStep` |
| 2 — Safety | §4 `EmailAction` + an approval gate |
| 2 — Memory | §5 `MemoryUpdate` |
| 3 — Validate & retry | §6 Read-only SQL guard with self-correction |
| 3 — Driving the loop | §7 An `AgentDecision`-driven agent |

The model reasons in language; **we get back a Pydantic object** we can execute,
inspect, reject, retry, and log.

## 0. Setup

We use the OpenAI SDK's **Structured Outputs** helper
`chat.completions.parse(..., response_format=Model)`, which returns a **validated
Pydantic instance** in `completion.choices[0].message.parsed`.
Needs `gpt-4o-mini` / `gpt-4o` (2024-08-06) or newer.

In [ ]:
# !pip install -q openai pydantic python-dotenv
import os, json
from dotenv import load_dotenv
load_dotenv("/home/amir/.env")
from typing import Optional, Literal
from pydantic import BaseModel, Field
from openai import OpenAI

OPENAI_MODEL = "gpt-4o-mini"
client = OpenAI()
print("OpenAI client ready — model:", OPENAI_MODEL)

## 1. The contract — a schema in, a validated object out

One tiny helper underlies the whole notebook: give it a Pydantic schema and a
request, get back a typed object. The API **constrains the model's output to the
schema** — no prose, no fences, no parsing.

In [ ]:
def ask(schema, user, system=None, model=OPENAI_MODEL):
    '''Return a validated instance of `schema` for the user request.'''
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": user})
    completion = client.chat.completions.parse(
        model=model, messages=messages, response_format=schema)
    return completion.choices[0].message.parsed     # a validated Pydantic instance


class AgentDecision(BaseModel):
    action: Literal["search", "calculate", "ask_user", "finish"]
    query: Optional[str] = None
    reason: str

d = ask(AgentDecision,
        "I need failure statistics for compressor units before I can answer. What next?")
print(type(d).__name__, "→")
print("  action:", d.action)
print("  query :", d.query)
print("  reason:", d.reason)
print("\nIt's a real object — branch on it:", "SEARCH" if d.action == "search" else d.action.upper())

## 2. Routing — dispatch to the right subsystem

The router's only job is to emit one typed decision. The `Literal` **closes the
world**: the model cannot return a route the controller doesn't handle.

In [ ]:
class RouteDecision(BaseModel):
    route: Literal["rag", "sql", "human", "general_answer"]
    confidence: float            # 0–1; OpenAI strict mode doesn't allow min/max, so no ge/le
    reason: str

ROUTER_SYS = (
    "You route a user request to one subsystem. "
    "rag = unstructured/document questions; sql = metrics/KPIs over a database; "
    "human = irreversible or risky actions needing approval; "
    "general_answer = small talk or things you can answer directly."
)

questions = [
    "What does the maintenance manual say about compressor seal replacement?",
    "What was our average ticket resolution time last quarter?",
    "Delete all customer records older than 5 years.",
    "Thanks, that's all for now!",
]

for q in questions:
    r = ask(RouteDecision, q, system=ROUTER_SYS)
    print(f"[{r.route:<15} conf={r.confidence:.2f}]  {q}")
    print(f"     reason: {r.reason}")

## 3. Planning — a plan is *data*, not prose

A structured plan can be shown, edited, re-ordered, and **executed** — a
free-text plan can only be read.

In [ ]:
class PlanStep(BaseModel):
    step_id: int
    tool: Literal["search", "sql", "python", "none"]
    instruction: str

class Plan(BaseModel):
    steps: list[PlanStep]

plan = ask(Plan,
           "Investigate why compressor units are failing and quantify the trend over time.",
           system="Produce a short ordered plan. Each step uses exactly one tool.")
for s in plan.steps:
    print(f"  {s.step_id}. [{s.tool:<6}] {s.instruction}")
print(f"\nThe executor can now loop over plan.steps ({len(plan.steps)} steps) and run each tool.")

## 4. Validation & safety — gate the side effect on a field

You can't gate prose. You *can* gate a field. The model proposes a typed
`EmailAction`; the controller enforces policy on the object **before** anything
is sent.

In [ ]:
class EmailAction(BaseModel):
    action: Literal["draft", "send"]
    recipient: str
    body: str
    requires_approval: bool

SAFETY_SYS = (
    "Turn the user's request into an EmailAction. "
    "Set requires_approval=true for any 'send' to a customer or external party."
)

def execute_email(a: EmailAction):
    # the guardrail lives on the OBJECT, before the side effect
    if a.action == "send" and a.requires_approval:
        return f"⛔ HELD for human approval: send to {a.recipient}"
    if a.action == "send":
        return f"✅ SENT to {a.recipient}"
    return f"📝 DRAFT saved for {a.recipient}"

for req in ["Draft a note to my teammate about the 3pm standup.",
            "Email the customer at acme@example.com that their refund was approved and send it."]:
    a = ask(EmailAction, req, system=SAFETY_SYS)
    print(f"request: {req}")
    print(f"  → action={a.action}  requires_approval={a.requires_approval}")
    print(f"  → {execute_email(a)}\n")

## 5. Memory updates — structured ops keep memory clean

The agent doesn't dump text into memory; it proposes a typed operation the
system can **drop** (`should_store=False`) or route by `memory_type`.

In [ ]:
class MemoryUpdate(BaseModel):
    should_store: bool
    memory_type: Literal["user_preference", "project_fact", "temporary_note"]
    content: str
    reason: str

MEM_SYS = ("Decide whether this user message is worth storing in long-term memory. "
           "Store durable preferences and project facts; skip small talk and one-off chatter.")

for msg in ["By the way, always give me distances in kilometers, not miles.",
            "ok cool thanks!",
            "Our compressor units are model CX-200, installed in 2019."]:
    m = ask(MemoryUpdate, msg, system=MEM_SYS)
    verb = f"STORE as {m.memory_type}" if m.should_store else "DROP"
    print(f"[{verb}]  {msg}")
    print(f"     reason: {m.reason}")

## 6. Validate → retry — hand the error back to the model

Structured output also lets us enforce **business rules** the schema alone can't
express, then **re-ask** when they're violated — the same "errors are messages
to the model" idea from the agent-loop deck, applied to the model's own output.

Here: a read-only SQL guard. We ask the model to *remove* records; the first
attempt is a destructive `DELETE`; the validator rejects it; we feed the error
back, and the model self-corrects to a safe `SELECT`.

In [ ]:
class SqlQuery(BaseModel):
    query: str

_BANNED = ["drop", "delete", "update", "insert", "alter", "truncate"]

def validate_readonly(q: SqlQuery):
    low = q.query.lower()
    hit = next((b for b in _BANNED if b in low), None)
    if hit:
        raise ValueError(
            f"Only read-only SELECT queries are allowed; '{hit.upper()}' is forbidden. "
            "Rewrite it as a SELECT that finds the rows instead of changing them."
        )

def ask_with_retry(schema, user, validate, system=None, max_retries=2):
    '''Parse, run a business-rule validator, and re-ask with the error on failure.'''
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": user})
    for attempt in range(1, max_retries + 2):
        obj = client.chat.completions.parse(
            model=OPENAI_MODEL, messages=messages, response_format=schema
        ).choices[0].message.parsed
        try:
            validate(obj)
            print(f"attempt {attempt}: ✅ accepted")
            return obj
        except ValueError as e:
            print(f"attempt {attempt}: ❌ {obj.query}")
            print(f"            rejected: {e}")
            # feed the rejected output + the reason back into the conversation
            messages.append({"role": "assistant", "content": obj.query})
            messages.append({"role": "user", "content": f"That was rejected: {e}"})
    print("gave up after retries")
    return obj

final = ask_with_retry(
    SqlQuery,
    "Remove all customers who have been inactive for more than 5 years from the customers table.",
    validate_readonly,
    system="You write SQL for a customers database.",
)
print("\nfinal query:", final.query)

## 7. Driving the loop with a decision schema

The payoff: the **controller** asks for one `AgentDecision` per turn and executes
the branch itself. The model reasons; the program decides what's allowed to
happen. Every decision is a typed object we print, gate, and log.

Tools are local and deterministic, so the trajectory you see is purely the
chain of **decisions**.

In [ ]:
# local, deterministic tools — the only network call is the LLM
_SEARCH = {"olympic": "The 2028 Summer Olympics will be held in Los Angeles, USA."}

def web_search(query: str) -> str:
    for k, v in _SEARCH.items():
        if k in query.lower():
            return v
    return f"No results for '{query}'."

def calculator(expression: str) -> str:
    import re
    if not re.fullmatch(r"[0-9+\-*/(). %]+", expression or ""):
        return f"Error: '{expression}' is not plain arithmetic."
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"


class LoopDecision(BaseModel):
    action: Literal["search", "calculate", "finish"]
    query: Optional[str] = Field(None, description="search query when action='search'")
    expression: Optional[str] = Field(None, description="arithmetic when action='calculate'")
    answer: Optional[str] = Field(None, description="final answer when action='finish'")
    reason: str

LOOP_SYS = (
    "You are an agent that works step by step. Each turn, return ONE decision. "
    "Use 'search' for facts you don't know, 'calculate' for any arithmetic, "
    "and 'finish' only when you can give the final answer. "
    "Base every step on the observations so far."
)

def run_decision_agent(task, max_turns=6):
    messages = [{"role": "system", "content": LOOP_SYS},
                {"role": "user", "content": task}]
    print(f"🎯 {task}\n" + "─" * 72)
    for turn in range(1, max_turns + 1):
        d = client.chat.completions.parse(
            model=OPENAI_MODEL, messages=messages, response_format=LoopDecision
        ).choices[0].message.parsed
        print(f"turn {turn}: action={d.action!r:<12} reason={d.reason}")
        messages.append({"role": "assistant", "content": d.model_dump_json()})

        if d.action == "finish":
            print("─" * 72 + f"\n✅ {d.answer}")
            return d.answer
        elif d.action == "search":
            obs = web_search(d.query or "")
            print(f"        🔧 web_search({d.query!r}) → {obs}")
        elif d.action == "calculate":
            obs = calculator(d.expression or "")
            print(f"        🔧 calculator({d.expression!r}) → {obs}")
        messages.append({"role": "user", "content": f"Observation: {obs}"})
    return "hit max_turns"

run_decision_agent(
    "Find the host city of the next Summer Olympics, then compute a 15% "
    "contingency on a budget of 320. Report the city and the contingency amount."
)

## Takeaways

- Every section asked the model for a **typed object**, not prose —
  `chat.completions.parse` gives back a validated Pydantic instance.
- That object is the **contract**: we branched on it (routing), executed it
  (planning, the loop), **gated** it before a side effect (safety), **dropped**
  it (memory), and **re-asked** when it broke a rule (validate → retry).
- Tool calling was just the first clause; routing, planning, safety, and memory
  are the same move pointed at different jobs.
- Because each decision is an object, the agent's behavior is **inspectable,
  gateable, and loggable** — that structured trace is what you evaluate next.

> Structured output turns the model's internal decision into a validated software
> object that the agent controller can execute, inspect, reject, retry, or log.